# Wilkinson 功率分配器

在本笔记本中，我们创建一个 [Wilkinson 功率分配器](https://www.microwaves101.com/encyclopedias/wilkinson-power-splitters)，它将输入信号分成两个等相输出信号。关于该电路的理论结果在参考 [1] 中介绍。在这里，我们将重现下图所示并在参考 [2] 中讨论的理想电路。在这个示例中，电路设计在 1 GHz 下工作。

In [ ]:
# standard imports
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.circuit import Circuit

rf.stylely()

In [ ]:
# frequency band
freq = rf.Frequency(start=0, stop=2, npoints=501, unit='GHz')

# characteristic impedance of the ports
z0_ports = 50

# resistor
R = 100
line_resistor = rf.media.DefinedGammaZ0(frequency=freq, z0=R)
resistor = line_resistor.resistor(R, name='resistor')

# branches
z0_branches = np.sqrt(2)*z0_ports
beta = freq.w/rf.constants.c
line_branches = rf.media.DefinedGammaZ0(frequency=freq, z0=z0_branches, gamma=0+beta*1j)

d = line_branches.theta_2_d(90, deg=True)  # @ 90°(lambda/4)@ 1 GHz is ~ 75 mm
branch1 = line_branches.line(d, unit='m', name='branch1')
branch2 = line_branches.line(d, unit='m', name='branch2')

# ports
port1 = Circuit.Port(freq, name='port1', z0=50)
port2 = Circuit.Port(freq, name='port2', z0=50)
port3 = Circuit.Port(freq, name='port3', z0=50)

# Connection setup
#┬Note that the order of appearance of the port in the setup is important
connections = [
           [(port1, 0), (branch1, 0), (branch2, 0)],
           [(port2, 0), (branch1, 1), (resistor, 0)],
           [(port3, 0), (branch2, 1), (resistor, 1)]
        ]

# Building the circuit
C = Circuit(connections)

可以通过可视化电路图形来检查电路设置（这需要可用的 python 包 `networkx`）。

In [ ]:
C.plot_graph(network_labels=True, edge_labels=True, port_labels=True, port_fontize=2)

让我们看看电路的散射参数：

In [ ]:
fig, (ax1,ax2) = plt.subplots(2, 1, sharex=True)
C.network.plot_s_db(ax=ax1, m=0, n=0,  lw=2)  # S11
C.network.plot_s_db(ax=ax1, m=1, n=1,  lw=2)  # S22
ax1.set_ylim(-90, 0)
C.network.plot_s_db(ax=ax2, m=1, n=0,  lw=2)  # S21
C.network.plot_s_db(ax=ax2, m=2, n=0,  ls='--', lw=2)  # S31
ax2.set_ylim(-4, 0)
fig.suptitle('Ideal Wilkinson Divider @ 1 GHz')

## 电流和电压

在之前的示例中，由于我们将 N 端口直接连接到外部端口，所以没有太多的内部端口。然而，在更复杂的场景中，电路可以有多个内部端口。

此外，较早的示例展示了串联连接组件的电路。随着电路复杂度的增加，一些组件以并联方式连接变得不可避免，这为计算电流和电压带来了新的挑战。

幸运的是，scikit-rf 很好地装备以处理这些复杂的电路场景。可以计算电路内部端口的电流和电压。

In [ ]:
power = [1,0,0]
phase = [0,0,0]
C.voltages(power, phase)  # or C2.currents(power, phase)

或者，你可以使用分路网络，在本例中是" T "网络，来创建只有成对连接的电路并执行电流和电压计算。虽然使用分路网络更接近构建电路的"经典"方式，但在代码可读性、复杂性和计算性能方面存在劣势。

In [ ]:
tee1 = line_branches.tee(name='tee1')
tee2 = line_branches.tee(name='tee2')
tee3 = line_branches.tee(name='tee3')

cnx = [
    [(port1, 0), (tee1, 0)],
    [(tee1, 1), (branch1, 0)],
    [(tee1, 2), (branch2, 0)],
    [(branch1, 1), (tee2, 0)],
    [(branch2, 1), (tee3, 0)],
    [(tee2, 2), (resistor, 0)],
    [(tee3, 2), (resistor, 1)],
    [(tee3, 1), (port3, 0)],
    [(tee2, 1), (port2, 0)],
]
C2 = Circuit(cnx)

结果图形稍微复杂一些：

In [ ]:
C2.plot_graph(network_labels=True, edge_labels=True, port_labels=True, port_fontize=2)

但结果是相同的：

In [ ]:
C.network == C2.network

In [ ]:
fig, (ax1,ax2) = plt.subplots(2, 1, sharex=True)
C2.network.plot_s_db(ax=ax1, m=0, n=0,  lw=2)  # S11
C2.network.plot_s_db(ax=ax1, m=1, n=1,  lw=2)  # S22
ax1.set_ylim(-90, 0)
C2.network.plot_s_db(ax=ax2, m=1, n=0,  lw=2)  # S21
C2.network.plot_s_db(ax=ax2, m=2, n=0,  ls='--', lw=2)  # S31
ax2.set_ylim(-4, 0)
fig.suptitle('Ideal Wilkinson Divider (2nd way) @ 1 GHz')

内部的电压和电流也是一致的。

In [ ]:
C2.voltages(power, phase)  # or C2.currents(power, phase)

你将在专门的示例中找到关于电压和电流计算的更多细节。